# Example of writing out the linear regression formula

Use the effects lookup to make a string with the formula in it.

## Code setup

In [2]:
import pandas as pd
import os

## Load in effects dictionary

In [3]:
df_effects = pd.read_csv(os.path.join('.', 'effects_linreg_allpatients_mrsleq2.csv'), index_col=0)

In [6]:
with pd.option_context('display.max_rows', 200): display(df_effects)

,precise_onset_known,afib_anticoagulant,prior_util,arrival_to_scan_time,thrombolysis,discharge_util,discharge_mrsleq2
onset_during_sleep,-0.65821,NaN,NaN,NaN,-0.11876,NaN,NaN
precise_onset_known,NaN,NaN,NaN,NaN,0.14176,NaN,NaN
atrial_fibrillation,NaN,0.63438,NaN,NaN,NaN,-0.02679,-0.02420
afib_anticoagulant,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prior_util,NaN,NaN,NaN,NaN,0.36069,0.55124,0.77955
age_80plus,NaN,NaN,-0.11836,NaN,-0.04444,-0.10816,-0.14663
stroke_severity_mild,NaN,NaN,NaN,NaN,-0.06370,0.12423,0.18851
stroke_severity_moderate,NaN,NaN,NaN,NaN,0.18796,-0.11943,-0.14345
stroke_severity_severe,NaN,NaN,NaN,NaN,-0.05008,-0.41916,-0.29924
arrival_to_scan_time,NaN,NaN,NaN,NaN,-0.00184,NaN,NaN


Rename stroke team columns for display:

In [5]:
# Pick out stroke team names:
stroke_team_names = df_effects.index[df_effects.index.str.isnumeric()].values
# Make a dictionary to relabel them as team_t:
dict_stroke_team_names = dict(zip(stroke_team_names, [f'team_{t}' for t in stroke_team_names]))
# Make a dictionary for the other index names to remain unchanged:
dict_index_names = dict(zip(df_effects.index, df_effects.index)) | dict_stroke_team_names

# Apply rename:
df_effects.index = df_effects.index.map(dict_index_names)

Check that rename worked:

In [6]:
df_effects.iloc[42:45]

,precise_onset_known,afib_anticoagulant,prior_util,arrival_to_scan_time,thrombolysis,discharge_util,discharge_mrsleq2
team_51,NaN,NaN,NaN,-14.92004,-0.02806,0.00063,0.06909
team_55,NaN,NaN,NaN,-2.85577,0.01118,0.08430,0.12274
team_56,NaN,NaN,NaN,-13.06753,-0.09347,0.01694,0.02346


## Function for making the formula

This uses the effects data to build up a formula as a text string.

In [7]:
def make_formula_string(df_effects, col, simplify_teams=False):   
    # Pick out values for this feature:
    s = df_effects[col].dropna()
    
    # Place formula in here:
    formula_bits = [f'{col} =']
    for i in s.index:
        s_here = f'{s.loc[i]:+.2f}'
        if i != 'const':
            # Add this name to the formula.
            s_here += f'×{i}'
        formula_bits.append(s_here)

    if simplify_teams:
        team_bits = []
        nonteam_bits = []
        for f in formula_bits:
            if 'team_' in f:
                team_bits.append(f)
            else:
                nonteam_bits.append(f)
        formula_str = ' '.join(nonteam_bits) + ' +lots of stroke teams'
    else:
        formula_str = ' '.join(formula_bits)
    # Place a space after each + and -:
    formula_str = formula_str.replace('+', '+ ')
    formula_str = formula_str.replace('-', '- ')
    return formula_str

Example for one feature:

In [8]:
# Make formula for this feature:
col = df_effects.columns[0]

formula_str = make_formula_string(df_effects, col)
print(formula_str)

precise_onset_known = - 0.66×onset_during_sleep + 0.66


Run for all features.

Features that don't use stroke team:

In [9]:
for col in df_effects.columns[:3]:
    print(make_formula_string(df_effects, col))

precise_onset_known = - 0.66×onset_during_sleep + 0.66
afib_anticoagulant = + 0.63×atrial_fibrillation + 0.04
prior_util = - 0.12×age_80plus + 0.90


Other features:

In [10]:
print(make_formula_string(df_effects, 'arrival_to_scan_time', simplify_teams=True))

arrival_to_scan_time = + 37.84 + lots of stroke teams


In [11]:
print(make_formula_string(df_effects, 'thrombolysis', simplify_teams=True))

thrombolysis = - 0.12×onset_during_sleep + 0.14×precise_onset_known + 0.36×prior_util - 0.04×age_80plus - 0.06×stroke_severity_mild + 0.19×stroke_severity_moderate - 0.05×stroke_severity_severe - 0.00×arrival_to_scan_time - 0.06 + lots of stroke teams


In [12]:
print(make_formula_string(df_effects, 'discharge_mrsleq2', simplify_teams=True))

discharge_mrsleq2 = - 0.02×atrial_fibrillation + 0.78×prior_util - 0.15×age_80plus + 0.19×stroke_severity_mild - 0.14×stroke_severity_moderate - 0.30×stroke_severity_severe + 0.10×thrombolysis - 0.15 + lots of stroke teams


In [13]:
# for x in range(6):
#     print(make_formula_string(df_effects, f'discharge_mrsleq{x}', simplify_teams=True))
#     print('')